In [2]:
import pandas as pd

train = pd.read_csv(
    'train.txt',
    sep=r'\s+',
    header=None
)

test = pd.read_csv(
    'test.txt',
    sep=r'\s+',
    header=None
)

train_labels = pd.read_csv(
    'train_labels.txt',
    sep=r'\s+',
    header=None
).squeeze()

test_labels = pd.read_csv(
    'test_labels.txt',
    sep=r'\s+',
    header=None
).squeeze()

In [3]:
X = pd.concat(
    [train, test],
    ignore_index=True
)

y = pd.concat(
    [train_labels, test_labels],
    ignore_index=True
)

print(X.shape)
print(y.shape)

(10299, 561)
(10299,)


In [4]:
number_of_activities = y.nunique()

print(number_of_activities)

6


In [5]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

print(round(X_scaled[0, 0], 2))

0.21


In [6]:
from sklearn.cluster import KMeans
from sklearn.metrics import (
    silhouette_score,
    calinski_harabasz_score,
    davies_bouldin_score
)

results = []

for n_clusters in range(2, 10):
    kmeans = KMeans(
        n_clusters=n_clusters,
        init='k-means++',
        random_state=42
    )

    cluster_labels = kmeans.fit_predict(X_scaled)

    results.append({
        'n_clusters': n_clusters,
        'silhouette': silhouette_score(
            X_scaled,
            cluster_labels
        ),
        'calinski_harabasz': calinski_harabasz_score(
            X_scaled,
            cluster_labels
        ),
        'davies_bouldin': davies_bouldin_score(
            X_scaled,
            cluster_labels
        )
    })

results = pd.DataFrame(results)

print(results.round(2))

   n_clusters  silhouette  calinski_harabasz  davies_bouldin
0           2        0.39            7880.81            1.07
1           3        0.32            5034.47            1.78
2           4        0.28            3668.02            2.05
3           5        0.26            2845.84            2.27
4           6        0.12            2504.69            2.53
5           7        0.10            2171.07            2.65
6           8        0.09            1890.89            2.62
7           9        0.08            1685.01            2.76


In [7]:
best_silhouette = results.loc[
    results['silhouette'].idxmax()
]

best_calinski = results.loc[
    results['calinski_harabasz'].idxmax()
]

best_davies = results.loc[
    results['davies_bouldin'].idxmin()
]

print(
    'Силуэт:',
    int(best_silhouette['n_clusters']),
    round(best_silhouette['silhouette'], 2)
)

print(
    'Калински — Харабас:',
    int(best_calinski['n_clusters']),
    round(best_calinski['calinski_harabasz'], 2)
)

print(
    'Дэвис — Болдин:',
    int(best_davies['n_clusters']),
    round(best_davies['davies_bouldin'], 2)
)

Силуэт: 2 0.39
Калински — Харабас: 2 7880.81
Дэвис — Болдин: 2 1.07


In [8]:
from sklearn.metrics import (
    homogeneity_score,
    completeness_score,
    adjusted_rand_score
)

kmeans_six = KMeans(
    n_clusters=6,
    init='random',
    random_state=42
)

clusters_six = kmeans_six.fit_predict(X_scaled)

homogeneity = homogeneity_score(y, clusters_six)
completeness = completeness_score(y, clusters_six)
ari = adjusted_rand_score(y, clusters_six)

print('Однородность:', round(homogeneity, 2))
print('Полнота:', round(completeness, 2))
print('ARI:', round(ari, 2))

Однородность: 0.54
Полнота: 0.58
ARI: 0.42


In [21]:
kmeans = KMeans(
    n_clusters=6,
    random_state=42
)

clusters = kmeans.fit_predict(X_scaled)

crosstab = pd.crosstab(y, clusters)

answers = crosstab.idxmax(axis=1) + 1

print(answers)

0
1    4
2    4
3    2
4    1
5    6
6    1
dtype: int32


In [18]:
main_clusters = cluster_table.idxmax(axis=1) + 1

print(main_clusters)

0
1    2
2    2
3    1
4    6
5    6
6    4
dtype: int32


In [22]:
from sklearn.cluster import KMeans
import pandas as pd

kmeans_two = KMeans(
    n_clusters=2,
    random_state=42
)

clusters_two = kmeans_two.fit_predict(X_scaled)

crosstab_two = pd.crosstab(
    y,
    clusters_two
)

print(crosstab_two)

col_0     0     1
0                
1         0  1722
2         8  1536
3         0  1406
4      1774     3
5      1906     0
6      1932    12


In [24]:
print(crosstab_two)

col_0     0     1
0                
1         0  1722
2         8  1536
3         0  1406
4      1774     3
5      1906     0
6      1932    12


In [25]:
from sklearn.metrics import completeness_score

score = completeness_score(y, clusters_two)
print(round(score, 2))

0.98


In [26]:
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import completeness_score

agglo = AgglomerativeClustering(
    n_clusters=2,
    linkage='ward'
)

clusters_agglo = agglo.fit_predict(X_scaled)

completeness_agglo = completeness_score(y, clusters_agglo)

print(completeness_agglo)
print(round(completeness_agglo, 2))

0.9999999999999993
1.0


In [27]:
print('K-means:', round(completeness_score(y, clusters_two), 2))
print('Agglomerative:', round(completeness_agglo, 2))

K-means: 0.98
Agglomerative: 1.0


In [30]:
x = np.array([1, 2, 3, 4])       # замените на первый вектор
y = np.array([5, 6, 7, 8])       # замените на второй вектор

cov_matrix = np.cov(x, y)
answer = round(cov_matrix.sum(), 2)

print(cov_matrix)
print(answer)

[[1.66666667 1.66666667]
 [1.66666667 1.66666667]]
6.67


In [31]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

df = pd.read_csv('Country-data.csv')

features = df.drop(columns=['country'])

# 7.1
print(round(df['life_expec'].max(), 1))

# 7.2
scaler = StandardScaler()
X_scaled = scaler.fit_transform(features)

print(round(X_scaled[0, 0], 2))

# 7.3
corr = features.corr()
corr_without_diag = corr.where(~np.eye(corr.shape[0], dtype=bool))
print(round(corr_without_diag.abs().max().max(), 2))

# 7.4–7.5
pca = PCA()
X_pca = pca.fit_transform(X_scaled)

explained = pca.explained_variance_ratio_
print(np.cumsum(explained))
print(np.argmax(np.cumsum(explained) >= 0.90) + 1)
print(round(explained[0], 2))

# 7.6
pca_5 = PCA(n_components=5)
components = pca_5.fit_transform(X_scaled)

corr_components = np.corrcoef(components, rowvar=False)
count = np.sum(
    (np.abs(corr_components) > 0.1) &
    (~np.eye(5, dtype=bool))
)

print(count)

82.8
1.29
0.9
[0.4595174  0.63133365 0.76137624 0.87190786 0.94530998 0.97015232
 0.98275663 0.99256944 1.        ]
5
0.46
0


In [32]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

silhouette_scores = {}

for k in range(2, 11):
    model = KMeans(
        n_clusters=k,
        init='k-means++',
        random_state=1,
        n_init=10
    )
    labels = model.fit_predict(components)
    silhouette_scores[k] = silhouette_score(components, labels)

print(silhouette_scores)

best_k = max(silhouette_scores, key=silhouette_scores.get)
print(best_k)

kmeans = KMeans(
    n_clusters=best_k,
    init='k-means++',
    random_state=1,
    n_init=10
)

cluster_labels = kmeans.fit_predict(components)
df['cluster'] = cluster_labels

print(df.loc[df['child_mort'].idxmax()])
print(df.loc[df['gdpp'].idxmin()])

{2: 0.3044199499231816, 3: 0.3079769786519014, 4: 0.3235154307362652, 5: 0.32558063247176533, 6: 0.26711817927696424, 7: 0.2256950938124907, 8: 0.24037703530553411, 9: 0.27436633921353193, 10: 0.24031234938202223}
5
country       Haiti
child_mort    208.0
exports        15.3
health         6.91
imports        64.7
income         1500
inflation      5.45
life_expec     32.1
total_fer      3.33
gdpp            662
cluster           1
Name: 66, dtype: object
country       Burundi
child_mort       93.6
exports          8.92
health           11.6
imports          39.2
income            764
inflation        12.3
life_expec       57.7
total_fer        6.26
gdpp              231
cluster             1
Name: 26, dtype: object
